# Dataset de Morosidad Arancelaria — UNSTA

Este cuaderno documenta paso a paso la construcción del dataset analítico para el modelo de **predicción de morosidad** en el pago de cuotas arancelarias de la Universidad del Norte Santo Tomás de Aquino (UNSTA).

**Tipo de problema:** Clasificación binaria supervisada.  
**Granularidad:** una fila = un alumno × una cuota esperada.  
**Variable objetivo:** `target_no_pago` → 0 (pagó en término) | 1 (mora o no pagó).

## 1. Contexto del problema

La UNSTA necesita anticipar qué alumnos tienen mayor probabilidad de no abonar su cuota arancelaria en término. Esto permite a la administración:

- **Gestión proactiva de cobranza:** focalizar los esfuerzos de contacto en los alumnos con mayor riesgo.
- **Planificación financiera:** estimar con mayor precisión el flujo de ingresos esperado por período.
- **Intervención temprana:** identificar patrones de riesgo vinculados a variables demográficas, académicas o temporales.

### Fuente de datos

La información proviene de la base de datos MySQL 5.7 del sistema de gestión arancelaria (`eUNSTAv3`). Se cruzan tablas transaccionales de categorías arancelarias, recibos de pago, datos del alumno y datos personales.

## 2. Decisiones de diseño y reglas de negocio

### Ventana temporal
- **Incluidos:** períodos 2020 a 2026.

### Filtros aplicados
- Carreras con `cuota_hasta > 12` excluidas (0.1% del total; diplomaturas con esquema especial).
- Solo cuotas con `fecha_vencimiento <= CURDATE()` (cuotas ya vencidas al momento de la extracción).
- Un único registro activo por alumno+período: se toma `MAX(creado)` de `alumno_categoria_arancelaria`.

### Feature de contexto temporal
- `contexto_temporal = 'pandemia'` si período ∈ {2020, 2021}
- `contexto_temporal = 'post_pandemia'` en caso contrario

### Lógica del target (`target_no_pago`)

| Condición | target |
|---|---|
| No existe recibo para esa cuota (nunca pagó) | **1** |
| Existe recibo pero `nro_vencimiento > 0` (pagó tarde) | **1** |
| Existe recibo Y `nro_vencimiento = 0` (pagó en 1er vencimiento) | **0** |

### Lógica de fechas de vencimiento

| nro_cuota | Fecha de vencimiento |
|---|---|
| 1 a 10 | 15 del mes correspondiente (marzo=1 ... diciembre=10) del `periodo` |
| 11 | 15 de enero de `periodo + 1` |
| 12 | 15 de febrero de `periodo + 1` |

## 3. Tablas involucradas

| Tabla | Rol |
|---|---|
| `alumno_categoria_arancelaria` | Universo de cuotas esperadas por alumno y período |
| `categoria_arancelaria` | Parametriza el tipo y porcentaje de descuento |
| `recibo` | Registra cada pago con fecha y `nro_vencimiento` |
| `recibo_detalle` | Detalle del recibo: `nro_cuota` y `periodo` al que pertenece |
| `alumno` | Features académicas: carrera, sede, facultad |
| `persona` | Features demográficas: sexo, edad, discapacidad |
| `persona_domicilio` | Localidad y provincia del alumno |

## 4. Instalación de dependencias

**Ejecutar esta celda una sola vez** (la primera vez que se abra el notebook o al cambiar de entorno). Luego reiniciar el kernel y ejecutar el resto.

In [1]:
%pip install pandas pymysql sqlalchemy python-dotenv --quiet


Note: you may need to restart the kernel to use updated packages.


## 5. Imports y configuración

In [2]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

## 6. Conexión a MySQL

Las credenciales se cargan desde el archivo `.env` en la raíz del proyecto. Variables requeridas:
- `MYSQL_HOST`, `MYSQL_PORT`, `MYSQL_USER`, `MYSQL_PASSWORD`, `MYSQL_DB`

In [3]:
_env_path = Path.cwd().parent / '.env'
if _env_path.exists():
    load_dotenv(_env_path)
else:
    load_dotenv()

MYSQL_HOST = os.getenv('MYSQL_HOST', 'localhost')
MYSQL_PORT = int(os.getenv('MYSQL_PORT', '3306'))
MYSQL_USER = os.getenv('MYSQL_USER', 'root')
MYSQL_PASSWORD = os.getenv('MYSQL_PASSWORD', '')
MYSQL_DB = os.getenv('MYSQL_DB', 'eUNSTAv3')
MYSQL_TIMEOUT = int(os.getenv('MYSQL_CONNECT_TIMEOUT', '30'))

conn_str = (
    f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}'
    f'@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}'
    f'?connect_timeout={MYSQL_TIMEOUT}'
)
engine = create_engine(conn_str)

with engine.connect() as conn:
    result = conn.execute(text('SELECT 1'))
    result.fetchone()

print(f'Conexión OK → {MYSQL_DB} en {MYSQL_HOST}:{MYSQL_PORT} (usuario: {MYSQL_USER})')

Conexión OK → eUNSTAv3 en 10.172.0.12:3306 (usuario: arancelario_user)


## 7. Query maestra

Compatible con MySQL 5.7. Genera una fila por cada combinación alumno × cuota esperada, con features demográficas, académicas y la variable objetivo `target_no_pago`.

### Estructura de la subquery principal (`ce`)
- Parte de `alumno_categoria_arancelaria` cruzada con un generador de números 1-12.
- Filtra por `periodo BETWEEN 2020 AND 2026`, cuotas válidas (1-12) y registro más reciente (`MAX(creado)`).
- Calcula `fecha_vencimiento` según la lógica definida.
- Solo incluye cuotas ya vencidas (`HAVING fecha_vencimiento <= CURDATE()`).

### Joins
- `alumno` → features académicas.
- `persona` → features demográficas.
- `persona_domicilio` (LEFT) → localización.
- `categoria_arancelaria` → tipo de descuento.
- Subquery de pagos `pg` (LEFT) → para construir el target.

In [4]:
QUERY_MOROSIDAD = """
SELECT
    ce.id_alumno,
    a.id_persona,
    ce.periodo,
    ce.nro_cuota,
    ce.fecha_vencimiento,
    a.id_carrera,
    a.id_sede_academica,
    a.id_unidad_academica,
    a.id_facultad,
    ce.id_categoria_arancelaria,
    cat.porcentaje          AS pct_descuento,
    cat.categoria           AS cod_categoria,
    p.sexo,
    TIMESTAMPDIFF(YEAR, p.fecha_nacimiento, ce.fecha_vencimiento) AS edad_al_vencer,
    p.discapacidad,
    pd.id_localidad,
    pd.id_provincia,
    CASE
        WHEN ce.periodo IN (2020, 2021) THEN 'pandemia'
        ELSE 'post_pandemia'
    END AS contexto_temporal,
    pg.fecha_pago,
    pg.nro_venc_usado,
    DATEDIFF(pg.fecha_pago, ce.fecha_vencimiento) AS dias_atraso,
    CASE
        WHEN pg.id_alumno IS NULL  THEN 1
        WHEN pg.nro_venc_usado > 0 THEN 1
        ELSE                            0
    END AS target_no_pago
FROM (
    SELECT
        aca.id_alumno,
        aca.periodo,
        aca.id_categoria_arancelaria,
        n.nro_cuota,
        CASE n.nro_cuota
            WHEN 1  THEN DATE(CONCAT(aca.periodo,     '-03-15'))
            WHEN 2  THEN DATE(CONCAT(aca.periodo,     '-04-15'))
            WHEN 3  THEN DATE(CONCAT(aca.periodo,     '-05-15'))
            WHEN 4  THEN DATE(CONCAT(aca.periodo,     '-06-15'))
            WHEN 5  THEN DATE(CONCAT(aca.periodo,     '-07-15'))
            WHEN 6  THEN DATE(CONCAT(aca.periodo,     '-08-15'))
            WHEN 7  THEN DATE(CONCAT(aca.periodo,     '-09-15'))
            WHEN 8  THEN DATE(CONCAT(aca.periodo,     '-10-15'))
            WHEN 9  THEN DATE(CONCAT(aca.periodo,     '-11-15'))
            WHEN 10 THEN DATE(CONCAT(aca.periodo,     '-12-15'))
            WHEN 11 THEN DATE(CONCAT(aca.periodo + 1, '-01-15'))
            WHEN 12 THEN DATE(CONCAT(aca.periodo + 1, '-02-15'))
        END AS fecha_vencimiento
    FROM alumno_categoria_arancelaria aca
    JOIN (
        SELECT 1 AS nro_cuota UNION SELECT 2  UNION SELECT 3
        UNION SELECT 4        UNION SELECT 5  UNION SELECT 6
        UNION SELECT 7        UNION SELECT 8  UNION SELECT 9
        UNION SELECT 10       UNION SELECT 11 UNION SELECT 12
    ) n ON n.nro_cuota BETWEEN aca.cuota_desde AND aca.cuota_hasta
    WHERE aca.periodo BETWEEN 2020 AND 2026
      AND aca.cuota_desde BETWEEN 1 AND 12
      AND aca.cuota_hasta BETWEEN 1 AND 12
      AND aca.cuota_hasta >= aca.cuota_desde
      AND aca.creado = (
          SELECT MAX(aca2.creado)
          FROM alumno_categoria_arancelaria aca2
          WHERE aca2.id_alumno = aca.id_alumno
            AND aca2.periodo   = aca.periodo
      )
    HAVING fecha_vencimiento <= CURDATE()
) ce
JOIN alumno a
    ON a.id_alumno = ce.id_alumno
JOIN persona p
    ON p.id_persona = a.id_persona
LEFT JOIN persona_domicilio pd
    ON pd.id_persona = a.id_persona
    AND pd.actual    = 1
JOIN categoria_arancelaria cat
    ON cat.id_categoria_arancelaria = ce.id_categoria_arancelaria
LEFT JOIN (
    SELECT
        r.id_alumno,
        rd.nro_cuota,
        rd.periodo,
        MIN(r.fecha)           AS fecha_pago,
        MIN(r.nro_vencimiento) AS nro_venc_usado
    FROM recibo r
    JOIN recibo_detalle rd ON rd.id_recibo = r.id_recibo
    WHERE rd.anulado = 0
    GROUP BY r.id_alumno, rd.nro_cuota, rd.periodo
) pg ON  pg.id_alumno = ce.id_alumno
     AND pg.nro_cuota  = ce.nro_cuota
     AND pg.periodo     = ce.periodo
ORDER BY ce.id_alumno, ce.periodo, ce.nro_cuota
"""

print('Query definida correctamente.')
print(f'Longitud: {len(QUERY_MOROSIDAD)} caracteres')

Query definida correctamente.
Longitud: 3403 caracteres


## 8. Extracción del dataset

Se ejecuta la query contra MySQL y se carga directamente en un DataFrame de pandas. Dependiendo del volumen, puede tardar unos minutos.

In [5]:
%%time

df = pd.read_sql(text(QUERY_MOROSIDAD), engine)

print(f'Filas extraídas:   {len(df):,}')
print(f'Columnas:          {df.shape[1]}')
print(f'Columnas (lista):  {df.columns.tolist()}')

Filas extraídas:   532,621
Columnas:          22
Columnas (lista):  ['id_alumno', 'id_persona', 'periodo', 'nro_cuota', 'fecha_vencimiento', 'id_carrera', 'id_sede_academica', 'id_unidad_academica', 'id_facultad', 'id_categoria_arancelaria', 'pct_descuento', 'cod_categoria', 'sexo', 'edad_al_vencer', 'discapacidad', 'id_localidad', 'id_provincia', 'contexto_temporal', 'fecha_pago', 'nro_venc_usado', 'dias_atraso', 'target_no_pago']
CPU times: total: 19.7 s
Wall time: 3min 1s


## 9. Validación rápida del dataset

In [6]:
print('=== Tipos de datos ===')
print(df.dtypes)
print()
print('=== Primeras 5 filas ===')
df.head()

=== Tipos de datos ===
id_alumno                     int64
id_persona                    int64
periodo                       int64
nro_cuota                     int64
fecha_vencimiento            object
id_carrera                   object
id_sede_academica             int64
id_unidad_academica           int64
id_facultad                  object
id_categoria_arancelaria      int64
pct_descuento               float64
cod_categoria                object
sexo                         object
edad_al_vencer              float64
discapacidad                  int64
id_localidad                float64
id_provincia                float64
contexto_temporal            object
fecha_pago                   object
nro_venc_usado              float64
dias_atraso                 float64
target_no_pago                int64
dtype: object

=== Primeras 5 filas ===


,id_alumno,id_persona,periodo,nro_cuota,fecha_vencimiento,id_carrera,id_sede_academica,id_unidad_academica,id_facultad,id_categoria_arancelaria,pct_descuento,cod_categoria,sexo,edad_al_vencer,discapacidad,id_localidad,id_provincia,contexto_temporal,fecha_pago,nro_venc_usado,dias_atraso,target_no_pago
0,66,66,2023,1,2023-03-15,18,1,3,I,2198,0.0,ACSVN,F,55.0,0,NaN,NaN,post_pandemia,None,NaN,NaN,1
1,66,66,2023,2,2023-04-15,18,1,3,I,2198,0.0,ACSVN,F,55.0,0,NaN,NaN,post_pandemia,None,NaN,NaN,1
2,66,66,2023,3,2023-05-15,18,1,3,I,2198,0.0,ACSVN,F,55.0,0,NaN,NaN,post_pandemia,None,NaN,NaN,1
3,66,66,2023,4,2023-06-15,18,1,3,I,2198,0.0,ACSVN,F,55.0,0,NaN,NaN,post_pandemia,None,NaN,NaN,1
4,66,66,2023,5,2023-07-15,18,1,3,I,2198,0.0,ACSVN,F,55.0,0,NaN,NaN,post_pandemia,None,NaN,NaN,1


In [7]:
print('=== Valores nulos por columna ===')
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
pd.DataFrame({'nulos': nulos, 'pct': nulos_pct}).query('nulos > 0').sort_values('pct', ascending=False)

=== Valores nulos por columna ===


,nulos,pct
id_localidad,468385,87.94
id_provincia,467844,87.84
fecha_pago,178846,33.58
nro_venc_usado,178846,33.58
dias_atraso,178846,33.58
edad_al_vencer,7906,1.48
sexo,60,0.01


In [8]:
print('=== Distribución del target ===')
target_dist = df['target_no_pago'].value_counts()
target_pct = df['target_no_pago'].value_counts(normalize=True).mul(100).round(2)

resumen_target = pd.DataFrame({
    'cantidad': target_dist,
    'porcentaje': target_pct
})
resumen_target.index = resumen_target.index.map({0: '0 - Pagó en término', 1: '1 - Mora / No pagó'})
print(resumen_target)
print(f'\nRatio mora/pago: {target_dist[1] / target_dist[0]:.2f}:1')

=== Distribución del target ===
                     cantidad  porcentaje
target_no_pago                           
1 - Mora / No pagó     389877        73.2
0 - Pagó en término    142744        26.8

Ratio mora/pago: 2.73:1


In [9]:
print('=== Distribución por período ===')
df.groupby('periodo')['target_no_pago'].agg(['count', 'sum', 'mean']).rename(
    columns={'count': 'total_cuotas', 'sum': 'cuotas_mora', 'mean': 'tasa_mora'}
).round(4)

=== Distribución por período ===


,total_cuotas,cuotas_mora,tasa_mora
periodo,,,
2020,71791,57051,0.7947
2021,79143,61428,0.7762
2022,87421,65609,0.7505
2023,97892,70542,0.7206
2024,95536,67596,0.7075
2025,99696,66854,0.6706
2026,1142,797,0.6979


In [10]:
print('=== Distribución por contexto temporal ===')
df.groupby('contexto_temporal')['target_no_pago'].agg(['count', 'sum', 'mean']).rename(
    columns={'count': 'total_cuotas', 'sum': 'cuotas_mora', 'mean': 'tasa_mora'}
).round(4)

=== Distribución por contexto temporal ===


,total_cuotas,cuotas_mora,tasa_mora
contexto_temporal,,,
pandemia,150934,118479,0.785
post_pandemia,381687,271398,0.711


In [11]:
print('=== Alumnos únicos y cuotas por alumno ===')
n_alumnos = df['id_alumno'].nunique()
cuotas_por_alumno = df.groupby('id_alumno').size()
print(f'Alumnos únicos:         {n_alumnos:,}')
print(f'Cuotas/alumno (media):  {cuotas_por_alumno.mean():.1f}')
print(f'Cuotas/alumno (mediana):{cuotas_por_alumno.median():.1f}')
print(f'Cuotas/alumno (min):    {cuotas_por_alumno.min()}')
print(f'Cuotas/alumno (max):    {cuotas_por_alumno.max()}')

=== Alumnos únicos y cuotas por alumno ===
Alumnos únicos:         25,904
Cuotas/alumno (media):  20.6
Cuotas/alumno (mediana):12.0
Cuotas/alumno (min):    1
Cuotas/alumno (max):    633


## 10. Exportación a CSV

Se guarda el dataset completo en `training/data/` para su uso posterior en EDA y entrenamiento de modelos.

In [12]:
out_dir = Path('../training/data')
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / 'dataset_morosidad.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')

size_mb = csv_path.stat().st_size / (1024 * 1024)
print(f'Dataset exportado: {csv_path.resolve()}')
print(f'Tamaño:            {size_mb:.2f} MB')
print(f'Filas:             {len(df):,}')
print(f'Columnas:          {df.shape[1]}')

Dataset exportado: C:\Users\Chelo\OneDrive\Documents\Cursos\Especializacion en IA\Proyecto Final\ia-predictiva-universitaria\training\data\dataset_morosidad.csv
Tamaño:            51.03 MB
Filas:             532,621
Columnas:          22


## 13. Descomposición de la mora — Análisis de severidad

El target binario actual (`target_no_pago`) agrupa tres situaciones muy diferentes bajo "mora":

| Situación | Comportamiento real | ¿Es mora grave? |
|---|---|---|
| `fecha_pago` es NULL | **Nunca pagó** | Sí — mora real |
| `nro_venc_usado > 0` y `dias_atraso` pequeño | **Pagó tarde pero pagó** (mora leve) | No necesariamente |
| `nro_venc_usado > 0` y `dias_atraso` grande | **Pagó muy tarde** (mora media) | Depende del umbral |

Un 78% de "mora" probablemente incluye una gran proporción de alumnos que **sí pagan, pero unos días después del vencimiento**. Esto es operativamente normal en cualquier universidad.

### Objetivo de esta sección

Descomponer la mora en categorías de severidad para:
1. Entender la composición real del 78%.
2. Redefinir el target según lo que la UNSTA realmente necesita predecir.
3. Obtener un modelo más útil y con métricas más honestas.

In [13]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('../training/data/dataset_morosidad.csv'))
print(f'Dataset cargado: {len(df):,} filas')

nunca_pago = df['fecha_pago'].isna() & (df['target_no_pago'] == 1)
pago_tarde = df['fecha_pago'].notna() & (df['nro_venc_usado'] > 0)
pago_en_termino = df['fecha_pago'].notna() & (df['nro_venc_usado'] == 0)

print(f'\n=== Descomposición del target ===')
print(f'Pagó en término (target=0):  {pago_en_termino.sum():>8,}  ({pago_en_termino.mean()*100:.1f}%)')
print(f'Pagó tarde (target=1):       {pago_tarde.sum():>8,}  ({pago_tarde.mean()*100:.1f}%)')
print(f'Nunca pagó (target=1):       {nunca_pago.sum():>8,}  ({nunca_pago.mean()*100:.1f}%)')
print(f'                             --------')
print(f'Total:                       {len(df):>8,}')

C:\Users\Chelo\AppData\Local\Temp\ipykernel_8484\226269414.py:4: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(Path('../training/data/dataset_morosidad.csv'))


Dataset cargado: 532,621 filas

=== Descomposición del target ===
Pagó en término (target=0):   142,744  (26.8%)
Pagó tarde (target=1):        211,031  (39.6%)
Nunca pagó (target=1):        178,846  (33.6%)
                             --------
Total:                        532,621


In [14]:
print('=== Distribución de días de atraso (solo quienes pagaron tarde) ===')
mora_con_pago = df[pago_tarde]['dias_atraso'].dropna()

print(f'\nEstadísticas:')
print(mora_con_pago.describe().round(1))

print(f'\n=== Distribución por rangos de atraso ===')
bins = [-1, 0, 7, 15, 30, 60, 90, 180, 365, float('inf')]
labels = ['0 días', '1-7 días', '8-15 días', '16-30 días', '31-60 días',
          '61-90 días', '91-180 días', '181-365 días', '>365 días']

df_tardios = df[pago_tarde].copy()
df_tardios['rango_atraso'] = pd.cut(df_tardios['dias_atraso'], bins=bins, labels=labels)

rango_dist = df_tardios['rango_atraso'].value_counts().sort_index()
rango_pct = (rango_dist / rango_dist.sum() * 100).round(2)

resumen_rangos = pd.DataFrame({
    'cantidad': rango_dist,
    'porcentaje': rango_pct,
    'pct_acumulado': rango_pct.cumsum().round(2)
})
print(resumen_rangos)

=== Distribución de días de atraso (solo quienes pagaron tarde) ===

Estadísticas:
count    211031.0
mean          3.8
std          56.6
min        -365.0
25%          -5.0
50%          -3.0
75%           1.0
max        1545.0
Name: dias_atraso, dtype: float64

=== Distribución por rangos de atraso ===
              cantidad  porcentaje  pct_acumulado
rango_atraso                                     
0 días            6795       11.07          11.07
1-7 días         18903       30.81          41.88
8-15 días         7848       12.79          54.67
16-30 días       10932       17.82          72.49
31-60 días        7436       12.12          84.61
61-90 días        3336        5.44          90.05
91-180 días       3569        5.82          95.87
181-365 días      1725        2.81          98.68
>365 días          811        1.32         100.00


In [15]:
import numpy as np

def clasificar_mora(row):
    if pd.isna(row['fecha_pago']):
        return 'sin_pago'
    if row['nro_venc_usado'] == 0:
        return 'en_termino'
    dias = row['dias_atraso']
    if pd.isna(dias) or dias <= 15:
        return 'mora_leve'
    elif dias <= 60:
        return 'mora_media'
    else:
        return 'mora_grave'

df['tipo_mora'] = df.apply(clasificar_mora, axis=1)

print('=== Distribución por tipo de mora ===')
tipo_dist = df['tipo_mora'].value_counts()
tipo_pct = (tipo_dist / len(df) * 100).round(2)

resumen_tipo = pd.DataFrame({
    'cantidad': tipo_dist,
    'porcentaje': tipo_pct
}).loc[['en_termino', 'mora_leve', 'mora_media', 'mora_grave', 'sin_pago']]

print(resumen_tipo)
print(f'\nTotal: {len(df):,}')

=== Distribución por tipo de mora ===
            cantidad  porcentaje
tipo_mora                       
en_termino    142744       26.80
mora_leve     183222       34.40
mora_media     18368        3.45
mora_grave      9441        1.77
sin_pago      178846       33.58

Total: 532,621


### Propuesta de targets alternativos

Según los resultados anteriores, podemos redefinir el target de formas más útiles:

**Opción A — Target estricto (mora real):**
Solo "nunca pagó" = 1, todo lo demás = 0. Predice impago absoluto.

**Opción B — Target de pago en el próximo mes (mora leve como foco):**
Pagó en término + mora leve (≤15 días) = 0, mora media + grave + sin pago = 1. Responde a: "¿este alumno me va a pagar dentro del mes?"

**Opción C — Target multi-clase (4 categorías):**
`en_termino` / `mora_leve` / `mora_media_grave` / `sin_pago`. Permite un modelo ordinal o multi-clase que capture la gradualidad.

In [16]:
print('=== Simulación de targets alternativos ===\n')

# Opción A: solo "nunca pagó" es mora
df['target_A_impago_real'] = (df['tipo_mora'] == 'sin_pago').astype(int)
pct_a = df['target_A_impago_real'].mean() * 100
print(f'Opción A (impago real):     {pct_a:.1f}% mora  /  {100-pct_a:.1f}% pagó')

# Opción B: mora leve se considera "va a pagar" (≤15 días)
df['target_B_pago_mes'] = df['tipo_mora'].isin(['mora_media', 'mora_grave', 'sin_pago']).astype(int)
pct_b = df['target_B_pago_mes'].mean() * 100
print(f'Opción B (pago en el mes):  {pct_b:.1f}% no paga a tiempo  /  {100-pct_b:.1f}% paga dentro del mes')

# Opción C: multi-clase
df['target_C_multiclase'] = df['tipo_mora'].map({
    'en_termino': 0,
    'mora_leve': 1,
    'mora_media': 2,
    'mora_grave': 2,
    'sin_pago': 3
})
print(f'\nOpción C (multi-clase):')
print(df['target_C_multiclase'].value_counts().sort_index().rename({
    0: '0 - En término',
    1: '1 - Mora leve (≤15d)',
    2: '2 - Mora media/grave (>15d)',
    3: '3 - Nunca pagó'
}))

=== Simulación de targets alternativos ===

Opción A (impago real):     33.6% mora  /  66.4% pagó
Opción B (pago en el mes):  38.8% no paga a tiempo  /  61.2% paga dentro del mes

Opción C (multi-clase):
target_C_multiclase
0 - En término                 142744
1 - Mora leve (≤15d)           183222
2 - Mora media/grave (>15d)     27809
3 - Nunca pagó                 178846
Name: count, dtype: int64


In [17]:
print('=== Distribución por tipo de mora y contexto temporal ===\n')
cross = pd.crosstab(
    df['contexto_temporal'],
    df['tipo_mora'],
    margins=True,
    normalize='index'
).mul(100).round(2)

cross = cross[['en_termino', 'mora_leve', 'mora_media', 'mora_grave', 'sin_pago']]
print(cross)

print('\n=== Distribución por tipo de mora y período ===\n')
cross_periodo = pd.crosstab(
    df['periodo'],
    df['tipo_mora'],
    normalize='index'
).mul(100).round(2)
cross_periodo = cross_periodo[['en_termino', 'mora_leve', 'mora_media', 'mora_grave', 'sin_pago']]
print(cross_periodo)

=== Distribución por tipo de mora y contexto temporal ===

tipo_mora          en_termino  mora_leve  mora_media  mora_grave  sin_pago
contexto_temporal                                                         
pandemia                 21.5      36.08        4.59        2.21     35.62
post_pandemia            28.9      33.73        3.00        1.60     32.77
All                      26.8      34.40        3.45        1.77     33.58

=== Distribución por tipo de mora y período ===

tipo_mora  en_termino  mora_leve  mora_media  mora_grave  sin_pago
periodo                                                           
2020            20.53      37.63        4.62        2.47     34.75
2021            22.38      34.68        4.56        1.96     36.41
2022            24.95      37.40        3.72        2.21     31.72
2023            27.94      35.41        3.02        1.49     32.14
2024            29.25      32.52        2.53        1.43     34.27
2025            32.94      30.15        2.71   

## 14. Exclusión de categorías con reducción 100%

Alumnos con descuento del 100% no generan obligación de pago real. Si no existe recibo, es un tema administrativo, no mora. Incluirlos infla artificialmente la categoría "sin_pago".

In [18]:
print('=== Categorías arancelarias en el dataset ===\n')
cat_resumen = (
    df.groupby(['pct_descuento', 'cod_categoria'])
    .agg(
        filas=('target_no_pago', 'size'),
        sin_pago=('tipo_mora', lambda x: (x == 'sin_pago').sum()),
        tasa_sin_pago=('tipo_mora', lambda x: (x == 'sin_pago').mean())
    )
    .sort_values('filas', ascending=False)
    .reset_index()
)
cat_resumen['tasa_sin_pago'] = (cat_resumen['tasa_sin_pago'] * 100).round(2)
print(cat_resumen.to_string(index=False))

print(f'\n=== Resumen por porcentaje de descuento ===\n')
por_pct = (
    df.groupby('pct_descuento')
    .agg(
        filas=('target_no_pago', 'size'),
        sin_pago=('tipo_mora', lambda x: (x == 'sin_pago').sum()),
        tasa_sin_pago=('tipo_mora', lambda x: (x == 'sin_pago').mean())
    )
    .sort_values('filas', ascending=False)
)
por_pct['tasa_sin_pago'] = (por_pct['tasa_sin_pago'] * 100).round(2)
print(por_pct)

print(f'\n--- Filas con descuento = 100%: {(df["pct_descuento"] == 100).sum():,} ---')

=== Categorías arancelarias en el dataset ===

 pct_descuento cod_categoria  filas  sin_pago  tasa_sin_pago
           0.0         2CSVN 126083      9066           7.19
           0.0         1CSVN  93107     36370          39.06
           0.0         KCSVN  88781     69221          77.97
           0.0         ECSVN  39778     28556          71.79
           0.0         ACSVN  38746     14953          38.59
          70.0         RCSVN  26170      3560          13.60
          10.0         ACSR1  21810      1608           7.37
         100.0         ACSR0  20091      4504          22.42
          20.0         ACSR2  16309      1052           6.45
          50.0         ACSR5  12454       665           5.34
          30.0         ACSR3  10716       812           7.58
          60.0         2CSDM   8082       427           5.28
          40.0         ACSR4   7217       446           6.18
          60.0         ACSDM   6370      3318          52.09
          40.0         ACSD2   3468   

In [19]:
antes = len(df)
df_filtrado = df[df['pct_descuento'] < 100].copy()
despues = len(df_filtrado)

print(f'=== Filtro: excluir pct_descuento == 100% ===')
print(f'Antes:   {antes:,} filas')
print(f'Después: {despues:,} filas')
print(f'Eliminadas: {antes - despues:,} filas ({(antes - despues) / antes * 100:.2f}%)')

print(f'\n=== Nueva distribución del target original ===')
print(df_filtrado['target_no_pago'].value_counts(normalize=True).mul(100).round(2).rename({
    0: '0 - Pagó en término',
    1: '1 - Mora / No pagó'
}))

print(f'\n=== Nueva distribución por tipo de mora ===')
tipo_nuevo = df_filtrado['tipo_mora'].value_counts()
tipo_nuevo_pct = (tipo_nuevo / len(df_filtrado) * 100).round(2)
print(pd.DataFrame({
    'cantidad': tipo_nuevo,
    'porcentaje': tipo_nuevo_pct
}).loc[['en_termino', 'mora_leve', 'mora_media', 'mora_grave', 'sin_pago']])

=== Filtro: excluir pct_descuento == 100% ===
Antes:   532,621 filas
Después: 512,530 filas
Eliminadas: 20,091 filas (3.77%)

=== Nueva distribución del target original ===
target_no_pago
1 - Mora / No pagó     73.05
0 - Pagó en término    26.95
Name: proportion, dtype: float64

=== Nueva distribución por tipo de mora ===
            cantidad  porcentaje
tipo_mora                       
en_termino    138105       26.95
mora_leve     175373       34.22
mora_media     17140        3.34
mora_grave      7570        1.48
sin_pago      174342       34.02


In [20]:
out_dir = Path('../training/data')
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / 'dataset_morosidad.csv'
df_filtrado.to_csv(csv_path, index=False, encoding='utf-8')

size_mb = csv_path.stat().st_size / (1024 * 1024)
print(f'Dataset filtrado re-exportado: {csv_path.resolve()}')
print(f'Tamaño:   {size_mb:.2f} MB')
print(f'Filas:    {len(df_filtrado):,}')
print(f'Columnas: {df_filtrado.shape[1]}')

Dataset filtrado re-exportado: C:\Users\Chelo\OneDrive\Documents\Cursos\Especializacion en IA\Proyecto Final\ia-predictiva-universitaria\training\data\dataset_morosidad.csv
Tamaño:   56.83 MB
Filas:    512,530
Columnas: 26


In [21]:
print('=== Todas las categorías arancelarias (cod_categoria) en el dataset ===\n')
cat_info = (
    df.groupby(['cod_categoria', 'pct_descuento'])
    .agg(
        filas=('target_no_pago', 'size'),
        alumnos_unicos=('id_alumno', 'nunique'),
        tasa_sin_pago=('tipo_mora', lambda x: (x == 'sin_pago').mean())
    )
    .sort_values('filas', ascending=False)
    .reset_index()
)
cat_info['tasa_sin_pago'] = (cat_info['tasa_sin_pago'] * 100).round(2)
print(cat_info.to_string(index=False))

=== Todas las categorías arancelarias (cod_categoria) en el dataset ===

cod_categoria  pct_descuento  filas  alumnos_unicos  tasa_sin_pago
        2CSVN            0.0 126083           10381           7.19
        1CSVN            0.0  93107            6328          39.06
        KCSVN            0.0  88781            6519          77.97
        ECSVN            0.0  39778            3304          71.79
        ACSVN            0.0  38746            2654          38.59
        RCSVN           70.0  26170            1973          13.60
        ACSR1           10.0  21810            1556           7.37
        ACSR0          100.0  20091            1363          22.42
        ACSR2           20.0  16309            1080           6.45
        ACSR5           50.0  12454             792           5.34
        ACSR3           30.0  10716             728           7.58
        2CSDM           60.0   8082             955           5.28
        ACSR4           40.0   7217             474     

In [22]:
CATEGORIAS_EGRESADO = ['ECPVN', 'ECSDM', 'ECSR0', 'ECSR2', 'ECSR3', 'ECSR4', 'ECSVN', 'EESR1']

es_egresado = df['cod_categoria'].isin(CATEGORIAS_EGRESADO)

print(f'=== Filas con categoría de egresado en el dataset ===')
print(f'Total filas egresado: {es_egresado.sum():,} de {len(df):,} ({es_egresado.mean()*100:.2f}%)')
print(f'Alumnos únicos con categoría egresado: {df[es_egresado]["id_alumno"].nunique():,}')

print(f'\n=== Distribución de mora en filas de egresado ===')
egr = df[es_egresado]
if len(egr) > 0:
    tipo_egr = egr['tipo_mora'].value_counts()
    tipo_egr_pct = (tipo_egr / len(egr) * 100).round(2)
    print(pd.DataFrame({'cantidad': tipo_egr, 'porcentaje': tipo_egr_pct}))

    print(f'\n=== Detalle por cod_categoria de egresado ===')
    detalle = egr.groupby('cod_categoria').agg(
        filas=('target_no_pago', 'size'),
        sin_pago=('tipo_mora', lambda x: (x == 'sin_pago').sum()),
        tasa_sin_pago=('tipo_mora', lambda x: (x == 'sin_pago').mean())
    )
    detalle['tasa_sin_pago'] = (detalle['tasa_sin_pago'] * 100).round(2)
    print(detalle)
else:
    print('No hay filas con categoría de egresado en el dataset actual.')

=== Filas con categoría de egresado en el dataset ===


Total filas egresado: 39,778 de 532,621 (7.47%)
Alumnos únicos con categoría egresado: 3,304

=== Distribución de mora en filas de egresado ===
            cantidad  porcentaje
tipo_mora                       
sin_pago       28556       71.79
mora_leve       6349       15.96
en_termino      4240       10.66
mora_media       492        1.24
mora_grave       141        0.35

=== Detalle por cod_categoria de egresado ===
               filas  sin_pago  tasa_sin_pago
cod_categoria                                
ECSVN          39778     28556          71.79


## 11. Cierre de conexión

In [23]:
engine.dispose()
print('Conexión a MySQL cerrada.')

Conexión a MySQL cerrada.


## 12. Cierre metodológico

### Resumen del dataset generado

| Aspecto | Detalle |
|---|---|
| Granularidad | 1 fila = 1 alumno × 1 cuota esperada |
| Períodos | 2020 – 2026 |
| Target | `target_no_pago` (0 = pagó en término, 1 = mora/no pagó) |
| Desbalance esperado | ~78% mora / ~22% pago en término |

### Advertencia sobre desbalance de clases

El dataset tiene un fuerte desbalance (~78% clase 1, ~22% clase 0). **No usar accuracy como métrica principal.** Métricas adecuadas:
- AUC-ROC
- F1-Score
- Precision / Recall

Técnicas recomendadas: `class_weight='balanced'` en sklearn, SMOTE, o undersampling.

### Próximos pasos

1. **EDA completo:** distribuciones, correlaciones, efecto pandemia, análisis por carrera/sede.
2. **Feature engineering:** historial de mora del alumno en períodos anteriores (predictor más potente esperado).
3. **Modelos baseline:** Logistic Regression → Random Forest → XGBoost.
4. **Evaluación:** AUC-ROC, F1, matrices de confusión, análisis de calibración.